# Spare-It POC: Synthetic Image Generator

This file is used to generate copy paste images using the original dataset and taco. Base images will be selected from the original dataset, meaning they will be the bottom layer. Then, more images will be cut up by their annotations, where fragments belonging to a specific class will be taken and appended to the base image. This pasting step is performed a random amount of times per a given uniform variable. While the base image is being pasted on, the base json file that describes the class annotations is also added to by the pasted segments. This works by overlaying the annotations, in essence stacking the segments which in turn overwrites the annotation that was underneath it. The filepaths can be edited accordingly in the second cell.

In [ ]:
import os
import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from pycocotools.coco import COCO
from matplotlib import image
from PIL import Image
from scipy import ndimage
import math
from skimage import measure
from imantics import Polygons, Mask

In [ ]:
img_dir = './original_dataset/images/' # Source of image files from the original dataset
json_dir = './original_dataset/cocojson/' # Source of json files from the original dataset

taco_image = './taco_transformed/taco_image/images/' # Source of image files from the taco dataset
taco_json = './taco_transformed/taco_json/' # Source of json files from the taco dataset

synth_image = './copy_paste/synthetic_images' # Output directory for images
synth_mask = './copy_paste/masks/' # Output directory for jsons
synth_json = './copy_paste/synthetic_jsons/' # Output directory for masks

# For future datasets, append the filepath here
def json_to_img(json):
    fname = json.split('/')[-1].split('.json')[0]
    dataset = json.split('/')[1]
    if(dataset == 'taco_transformed'):
        return taco_image + fname + '.jpg'
    if(dataset == 'original_dataset'):
        return img_dir + fname + '.jpeg'
    else:
        return None

In [ ]:
# Methods for image cropping and masking
def crop(arr):
    objs = ndimage.find_objects(arr>0)
    if(len(objs) != 0):
        slice_x, slice_y = objs[0]
        return arr[slice_x, slice_y]
    return None
def cropc(arr):
    slice_x, slice_y, slice_z = ndimage.find_objects(arr>0)[0]
    return arr[slice_x, slice_y, slice_z]
def crop_coord(arr):
    slice_x, slice_y = ndimage.find_objects(arr>0)[0]
    return [slice_x, slice_y]
def apply_mask(image, mask):
    # Convert to numpy arrays
    mask = np.array(mask)
    # Convert grayscale image to RGB
    mask = np.stack((mask,)*3, axis=-1)
    # Multiply arrays
    resultant = image*mask
    return resultant
# Method for filenaming convention
# Finds current highest number filename in output directory and names the next file one more than it
def max_file(path):
    files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
    max = 0
    for f in files:
        s = f.split('.')
        if(int(s[0]) > max):
            max = int(s[0])
    return max + 1
# Method for sampling a normally distributed random variable for a given file dimension
def random_offset(dist):
    center = dist/2
    var = dist/10
    rand = math.ceil(np.random.normal(center, var, 1)[0])
    if(rand < dist/4 or rand > 3*dist/4):
        return math.ceil(center)
    else:
        return rand
# Converts a mask with multiple IDs to a list of binary masks with the related ID kept in a list
def mask_to_binary(mask):
    vals = np.unique(mask)
    z_index = np.where(vals == 0)
    vals = np.delete(vals, z_index)
    n = len(vals)
    binaries = []
    for i in range(n):
        binary = np.zeros(np.shape(mask))
        binary = np.where(mask == vals[i], 1, 0)
        binaries.append(binary)
    return binaries, vals
# Converts a mask with multiple IDs to COCO format annotations
def get_annotations(mask):
    bins, cids = mask_to_binary(mask)
    n = len(cids)
    annotations = []
    inc = 0
    for i in range(n):
        polygons = Mask(bins[i]).polygons().segmentation # Generates polygonal sets of coordinates for the mask
        for j in range(len(polygons)):
            ann = {}
            ann['id'] = inc
            inc += 1
            ann['image_id'] = 0
            ann['category_id'] = int(cids[i])
            polygon = polygons[j]
            seg = [polygon]
            ann['segmentation'] = seg
            annotations.append(ann)
    return annotations

In [ ]:
# Copy Paste Method
# Takes a list containing lists with a name of a json file and the category ID to be used with that json
# It also takes a base json file
# The part of each file that has its relevant category ID is pasted as chunks onto the base image with random positioning
# Returns the augmented image, json file, and mask
def copypaste(files, file2):
    coco2=COCO(file2) # COCO filetype for base image
    img2 = coco2.imgs[0] # Base image as described in json file
    imgpath2 = json_to_img(file2)
    image2 = np.array(Image.open(os.path.join(imgpath2))) # Array representing base image
    cat_ids = coco2.getCatIds()
    anns_ids2 = coco2.getAnnIds(imgIds=img2['id'], catIds=cat_ids, iscrowd=None)
    anns2 = coco2.loadAnns(anns_ids2) # List of annotations for base image
    mask2 = np.zeros((img2['height'],img2['width']))
    
    for i in range(len(anns2)):
        # Iterates over annotations and adds them to the mask
        # Each pixel is the ID of the category from the annotations
        temp = coco2.annToMask(anns2[i])*anns2[i]['category_id']
        mask2 = np.where(temp != 0, temp, mask2)
        
    for i in files:
        # Iterates over each file in the list of jsons
        file1 = i[0]
        cid = i[1]
        coco1=COCO(file1) # COCO filetype for currently pasted image
        img1 = coco1.imgs[0]
        imgpath1 = json_to_img(file1)
        image1 = np.array(Image.open(os.path.join(imgpath1)))
        anns_ids1 = coco1.getAnnIds(imgIds=img1['id'], catIds=cat_ids, iscrowd=None)
        anns1 = coco1.loadAnns(anns_ids1)
        mask1 = np.zeros((img1['height'],img1['width']))
        
        for i in range(len(anns1)):
            temp = coco1.annToMask(anns1[i])*anns1[i]['category_id']
            mask1 = np.where(temp != 0, temp, mask1)

        # Crops the image and mask for the current file being pasted to only include the selected category
        paste = np.where(mask1 == cid, 1, 0)
        mask_cropped = crop(paste)*cid
        image_cropped = cropc(apply_mask(image1, paste))

        # Gets a random offset for the x and y coordinate based on the base image file dimensions
        x_offset=random_offset(img2['width'])
        y_offset=random_offset(img2['height'])

        # Nested for loops that go pixel by pixel to place the pasted chunk onto the base image
        # Iterates over the dimensions of where the chunk will be pasted
        # Increments track where in the pasted chunk the loop is
        y_inc = 0
        for i in range(y_offset,y_offset+image_cropped.shape[0]):
            x_inc = 0
            if(i < img2['height']): # Ensures that the chunk is not pasted outside of the base image bounds
                for j in range(x_offset,x_offset+image_cropped.shape[1]):
                    if(mask_cropped[y_inc, x_inc] != 0 and j < img2['width']):
                        image2[i, j, :] = image_cropped[y_inc, x_inc, :]
                        mask2[i,j] = mask_cropped[y_inc, x_inc] # The mask is also updated with the pasted chunk
                    x_inc +=1
            y_inc += 1
    filenum = max_file(synth_image) # Generates serial number to be the name for all exported files 
    newImg = Image.fromarray(image2)
    newImg.save(synth_image + str(filenum) + '.jpeg') # Exports augmented image
    np.save(synth_mask + str(filenum), mask2) # Saves augmented mask as numpy array
    annotations = get_annotations(mask2) # Converts mask to COCO format annotations
    # Copies and exports the original json file with the annotations section updated
    with open(file2, 'r+') as file:
        template = json.load(file) 
    template['annotations'] = annotations
    with open(synth_json + str(filenum) + '.json', 'w') as f:
        json.dump(template, f, indent=4)

These operations accumulate the json directory/directories into dictionaries of sets, where the key is the class ID, and the value is a set containing the json filepaths that contain an annotation belonging to the class ID. The first dictionary files_by_id does this exactly for the original dataset. For files_by_id2, it performs a similar operation, but only selects files from the dataset that contain less than three annotations. This was done to create a pool of images that could be used as base images for copy paste to emulate an empty trash can. files_taco serves as the pool of jsons for the taco dataset. At the end, classes that have empty sets are removed. 

In [ ]:
# Loads all files into dictionary of sets
files = [f for f in os.listdir(json_dir) if os.path.isfile(os.path.join(json_dir, f))]
with open(json_dir + files[-1], 'r+') as file:
        template = json.load(file)
cats = template['categories']
categories = {}
for i in cats:
    categories[i['id']] = i['name']
files_by_id = {}
files_by_id2 = {}
files_taco = {}
for i in categories:
    files_by_id[i] = set()
    files_by_id2[i] = set()
    files_taco[i] = set()

for i in files:
    with open(json_dir + i, 'r+') as file:
        temp = json.load(file)
    annList = temp['annotations']
    for j in annList:
        cid = j['category_id']
        if(cid in categories):
            cats = temp['categories']
            
            cset = files_by_id[cid]
            cset.add(json_dir + i)
            files_by_id[cid] = cset
        
# The second dictionary records only files with less than 3 annotations
for i in files:
    with open(json_dir + i, 'r+') as file:
        temp = json.load(file)
    annList = temp['annotations']
    if(len(annList) < 3):
        for j in annList:
            cid = j['category_id']
            if(cid in categories):
                cats = temp['categories']
                
                cset = files_by_id2[cid]
                cset.add(json_dir + i)
                files_by_id2[cid] = cset

# Taco - to add further datasets, please copy this segment and provide it with a new dictionary
tfiles = [f for f in os.listdir(taco_json) if os.path.isfile(os.path.join(taco_json, f))]
for i in tfiles:
    with open(taco_json + i, 'r+') as file:
        temp = json.load(file)
    annList = temp['annotations']
    if(len(annList) < 3):
        for j in annList:
            cid = j['category_id']
            if(cid in categories):
                cats = temp['categories']
                
                cset = files_taco[cid]
                cset.add(taco_json + i)
                files_taco[cid] = cset

for i in files_by_id:
    cset = files_by_id[i]
    tset = files_taco[i]
    files_by_id[i] = cset.union(tset)


ids2 = []
for i in files_by_id2:
    if(len(files_by_id2[i]) > 0):
        ids2.append(i)

poplist = []
for i in files_by_id:
    if(len(files_by_id[i]) == 0):
        poplist.append(i)
for i in poplist:
    files_by_id.pop(i)

A probability distribution is generated to select underrepresented classes at a higher rate than the more prevalent classes. This is done by taking the difference in the number of images that contain each class from the class with the greatest number of occurrences. This produced reasonable results, with some notable exceptions, such as batteries, that have been manually adjusted after the fact due to their anomalous representation in the data.

In [ ]:
# Generates sampling probabilities to resolve class imbalance
target = 0
for i in files_by_id:
    if(target < len(files_by_id[i])):
        target = len(files_by_id[i])
to_do = {}
for i in files_by_id:
    to_do[i] = target - len(files_by_id[i])
to_do[16] = 50
to_do[38] = 50
to_do[76] = 50
sum = 0
for i in to_do:
    sum += to_do[i]
probs = np.asarray(list(to_do.values()))/sum # inverse distribution
plt.plot(probs)

This is the method that ultimately generates the copy paste images. The method balance_sampling() takes three parameters, an array representing the probability for each class, the number of images to be generated, and the upper bound for the uniform distribution of image segments that will be pasted. That is, setting r=30 will generate images that have between 1 and 30 fragments pasted on top of the original.  The method will produce three results, a synthetic image, the coco format json file containing the image's updated annotations, and the mask that was used to produce the annotations. The mask contains multiple classes, using the classes ID as a value per pixel. 

In [ ]:
# Generates copy paste images using inverse distribution
def balance_sampling(probs, k, r):
    for i in range(k):
        appends = []
        for i in range(random.randint(1, r)): # (1, 30) decides the range of random values for the number of image segments to be pasted
            rid1 = random.choices(tuple(to_do), probs)[0]
            rfile1 = random.choice(tuple(files_by_id[rid1]))
            appends.append([rfile1, rid1])
        rfile2 = random.choice(tuple(files_by_id2[random.choice(ids2)]))
        try:
            copypaste(appends, rfile2)
        except:
            print("Error")
balance_sampling(probs, 10000, 30)

In [ ]:
# Displays Generated Image For Specific ID
num = 1405 # Image ID
tj = synth_json + str(num) + '.json'
ti = np.array(Image.open(os.path.join(synth_image + str(num) + '.jpeg')))
plt.imshow(ti)
coco=COCO(tj)
catIds = coco.getCatIds()
cats = coco.loadCats(catIds)
ids = coco.getAnnIds()
cat_ids = coco.getCatIds()
img = coco.imgs[0]
anns_ids = coco.getAnnIds(imgIds=img['id'], catIds=cat_ids, iscrowd=None)
anns = coco.loadAnns(anns_ids)
coco.showAnns(anns)